<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of Analysis & Time Window

* **Unit of Analysis:** One row represents **one unique pseudonymized content item (`content_id`) for a specific client (`client_id`) evaluated within a single monthly observation panel**.
* **Primary Mid-Panel Evaluation Month:** `month = '2026-03'` (March 1, 2026 to March 31, 2026).
* **Data River & Grain Rule:** In this monthly panel, each `content_id` appears exactly once per client per month. Features reflect performance and metadata knowable up to the panel cut-off date.
* **Outcome Window:** The future target label (`is_declining_label`) evaluates performance across the subsequent 30-day window following the observation month.

In [5]:
import pandas as pd

raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

print("--- UNIT OF ANALYSIS VERIFICATION ---")
print(f"Total Dataset Rows: {len(df):,}")
print(f"Unique Content Units (content_id): {df['content_id'].nunique():,}")
print(f"Unique Clients (client_id): {df['client_id'].nunique():,}")
print(f"Grain Check: Is content_id unique in this slice? {df['content_id'].nunique() == len(df)}")


--- UNIT OF ANALYSIS VERIFICATION ---
Total Dataset Rows: 30,000
Unique Content Units (content_id): 30,000
Unique Clients (client_id): 32
Grain Check: Is content_id unique in this slice? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### 2. Field Schema Classification

Every field touched in this data contract is sorted strictly into one of four buckets:

1. **Features (Knowable at decision moment):**
   * `content_age_days`: Total days since the content item was originally created.
   * `days_since_last_update`: Days elapsed since the content item was last refreshed/edited.
   * `impressions_90d`: Aggregated impressions over the preceding 90-day window.
   * `avg_position`: Mean organic search rank position over the preceding 90-day window.
   * `ctr`: Click-through rate ($\frac{\text{clicks}}{\text{impressions}}$) over the preceding 90-day window.

2. **Target / Label (Observed outcome):**
   * `is_declining_label`: Binary label ($1$ if `trend_direction == "down"`, $0$ otherwise), representing an observed traffic decay outcome.

3. **Context (Entity identifiers & temporal anchors):**
   * `content_id`: Unique anonymized content identifier.
   * `client_id`: Unique anonymized client/domain identifier.
   * `trend_direction`: Raw categoric trend descriptor ("Down", "Stable", "Up").

4. **Excluded (Deliberately omitted & Why):**
   * `post_period_clicks` / `future_30d_impressions`: **Omitted due to direct Data Leakage.** Post-decision window outcomes are unknowable at the decision timestamp and would corrupt model training.
   * `raw_url_path` / `page_title_text`: **Omitted for PII & Anonymization compliance.** Anonymized hashes protect client privacy while maintaining entity identity.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Field schema verification and summary stats
feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
label_col = "is_declining_label"
context_cols = ["content_id", "client_id", "trend_direction"]

# Ensure target label exists
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("--- FIELD BUCKET VERIFICATION ---")
print("Features DataFrame Shape:", df[feature_cols].shape)
print("Target Distribution:\n", df[label_col].value_counts(normalize=True).rename({1: "Declining (1)", 0: "Stable/Up (0)"}))


--- FIELD BUCKET VERIFICATION ---
Features DataFrame Shape: (30000, 5)
Target Distribution:
 is_declining_label
Declining (1)    0.542067
Stable/Up (0)    0.457933
Name: proportion, dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3. Verification & Signal Availability Proofs

We execute three explicit verification checks on our panel data to prove our contract claims:
1. **Grain Check:** Confirming exact row-level uniqueness per `content_id`.
2. **Slice Metrics & Span:** Proving row counts, client distributions, and metric ranges.
3. **Availability Filter (`IS TRUE`):** Verifying the exact count and percentage of rows surviving signal readiness criteria (`days_since_last_update >= 90` and `impressions_90d >= 100`).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Prove Grain Uniqueness
total_rows = len(df)
unique_content = df["content_id"].nunique()
print(f"Query 1 [Grain Proof]: Total Rows = {total_rows:,} | Unique content_ids = {unique_content:,}")
assert total_rows == unique_content, "Grain failure: Duplicate content_id values found!"

# Query 2: Slice Row Count & Summary Range
print("\nQuery 2 [Slice Metrics]:")
print(f"Total Rows in Slice: {total_rows:,}")
print(f"Client Count: {df['client_id'].nunique():,}")
print(f"Content Age Range (Days): {df['content_age_days'].min()} to {df['content_age_days'].max()}")
print(f"Impressions 90d Range: {df['impressions_90d'].min():,} to {df['impressions_90d'].max():,}")

# Query 3: Availability Check using IS TRUE filter logic
# Filter: Stale (>=180 days) AND Visible (>=500 impressions)
availability_mask = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 100)
df["is_available_stale_visible"] = availability_mask

surviving_rows = df["is_available_stale_visible"].sum()
surviving_pct = (surviving_rows / total_rows) * 100

print("\nQuery 3 [Availability Check with IS TRUE logic]:")
print(f"Rows meeting (days_since_last_update >= 90 AND impressions_90d >= 100) IS TRUE: {surviving_rows:,} / {total_rows:,} ({surviving_pct:.2f}%)")

Query 1 [Grain Proof]: Total Rows = 30,000 | Unique content_ids = 30,000

Query 2 [Slice Metrics]:
Total Rows in Slice: 30,000
Client Count: 32
Content Age Range (Days): 90 to 564
Impressions 90d Range: 1 to 517,715

Query 3 [Availability Check with IS TRUE logic]:
Rows meeting (days_since_last_update >= 90 AND impressions_90d >= 100) IS TRUE: 8,119 / 30,000 (27.06%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Feature Construction & The Deliberate Leakage Trap

#### Feature Timeliness Definitions ("Available When?")
1. **`content_age_days`**: Knowable at decision time because creation timestamp is recorded upon publication.
2. **`days_since_last_update`**: Knowable at decision time because content management system logs edit timestamps deterministically.
3. **`impressions_90d`**: Knowable at decision time because search console logs historical impressions up through $T_0$.
4. **`avg_position`**: Knowable at decision time because search rank aggregation is logged over the pre-decision 90-day window.
5. **`ctr`**: Knowable at decision time because clicks and impressions are historical pre-decision facts.

---

### The Trap: Deliberate Target Leakage Demonstration
We intentionally inject a leaked feature (`future_decline_signal`) constructed directly from post-period trend behavior to show how leakage creates artificially perfect scores that fail in production.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
from sklearn.metrics import precision_score, roc_auc_score

# 1. Clean Feature Set (5 Features)
features_honest = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]

# 2. Honest Baseline Score (Simple Rule/Score based on Stale + Low CTR)
# Higher score = higher risk
df["honest_score"] = (df["days_since_last_update"] / 365.0) + (1.0 / (df["ctr"] + 0.01))
honest_auc = roc_auc_score(df["is_declining_label"], df["honest_score"])

print("--- HONEST MODEL / SCORE EVALUATION ---")
print(f"Honest Feature ROC-AUC Score: {honest_auc:.4f}")

# 3. TRAP: Inject Leaked Column derived directly from target
# Adding noise to target simulates a post-period future measurement
np.random.seed(42)
df["leaked_future_signal"] = df["is_declining_label"] * 0.9 + np.random.normal(0, 0.1, size=len(df))

leaked_auc = roc_auc_score(df["is_declining_label"], df["leaked_future_signal"])
print("\n--- LEAKAGE TRAP DEMONSTRATION ---")
print(f"Leaked Feature ROC-AUC Score: {leaked_auc:.4f} (Artificially perfect!)")

# 4. REMOVE THE TRAP
df.drop(columns=["leaked_future_signal"], inplace=True)
print("\n✅ Leakage column successfully dropped! Restored honest baseline dataset.")

--- HONEST MODEL / SCORE EVALUATION ---
Honest Feature ROC-AUC Score: 0.4957

--- LEAKAGE TRAP DEMONSTRATION ---
Leaked Feature ROC-AUC Score: 1.0000 (Artificially perfect!)

✅ Leakage column successfully dropped! Restored honest baseline dataset.


### 5. Data Limitations & Known Boundaries

* **GSC Unbalanced History:** Search Console data only retains historical granularity up to 16 months. Pages with long evergreen histories suffer from left-censoring on early impression trends.
* **Anonymized Aggregations:** Aggregate 90-day windows smooth out acute, short-term algorithmic volatility (e.g., core updates occurring in a 7-day span).
* **Missing Off-Page Signals:** The dataset lacks backlink authority metrics, competitor movements, and search intent shifts, meaning performance predictions depend strictly on internal observable signals.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.